# Higher Education by Campus and College
The previous page covers the three systems as a whole; this one splits them into **UC campuses**, **CSU campuses** and **community colleges** (and community college districts). The source and caveats are the same: the U.S. Department of Education's IPEDS finance survey, FY2004-2023 (`scripts/fetch_ipeds.py`).

**Read this first**
- **UC totals include medical centers.** UCLA, UCSF, UC San Diego, UC Irvine and UC Davis run hospitals and clinics, so their total spending is far above the others. The per-student charts use *core* spending (instruction, research, public service, academic support, student services, administration and plant operation), which leaves out hospitals, housing and dining, and financial aid. UC San Francisco (health sciences only, about 1,200 students) and the UC College of the Law (SF) are left out of the per-student charts because their student counts are small and specialized.
- **Community colleges file as colleges or as districts.** Los Angeles's nine colleges each file separately, but a few districts (Antelope Valley, El Camino, Feather River, Mt. San Jacinto) file once for the whole district, so a name here is the unit that filed and comparing units of different structure is rough. Colleges with fewer than 3,000 full-time-equivalent students are left out of the rankings. State money in a single year can exceed a college's core spending (Butte College in FY2023, for example) when it includes capital or one-time grants.
- **Student counts (FTE)** cover credit instruction only. Property taxes appear only for community colleges.
- Dollars are real (2026 dollars) unless a chart says otherwise. Function and benefit caveats from the previous page apply; the core-spending total also changes definition at FY2010 (a few percent), marked by a dotted line.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, line_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
d = pd.read_csv(PROCESSED / "highered_institutions.csv")
mac = pd.read_csv(PROCESSED / "macro.csv").set_index("fy")
d["factor"] = d.fy.map(mac.cpi_factor)
HE = "Source: U.S. Dept. of Education, IPEDS finance survey"
def short(n):
    for a, b in (("University of California College of the Law-San Francisco", "UC Law, SF"), ("University of California-", "UC "), ("California State University-", "CSU "),
                 ("California State University Maritime Academy", "CSU Maritime"), ("California State Polytechnic University-", "Cal Poly "),
                 ("California Polytechnic State University-San Luis Obispo", "Cal Poly San Luis Obispo"), (" University", "")):
        n = n.replace(a, b)
    return n
d["name"] = d.institution.map(short)
for k in ("total", "core", "state", "local_tax", "tuition"):
    d[k + "_real_b"] = d[k + "_k"] * d.factor / 1e6          # $ billions, base-year dollars
    d[k + "_pf"] = d[k + "_k"] * d.factor * 1000 / d.fte      # $ per FTE student, base-year dollars
uc, csu, cc = (d[d.system == s] for s in ("UC", "CSU", "Community colleges"))
def lines(df, col, title, sub, names, ytitle, fmt="$%{y:,.0f}"):
    fig = go.Figure()
    for i, n in enumerate(names):
        z = df[df.name == n]
        fig.add_trace(go.Scatter(x=z.fy, y=z[col], name=n, mode="lines+markers", line=dict(color=PALETTE[i % len(PALETTE)]), marker=dict(size=4), hovertemplate=fmt))
    fig.update_layout(yaxis_title=ytitle, xaxis_title="Fiscal year (ending June 30)")
    fig.add_vline(x=2009.5, line_dash="dot", line_color="grey", annotation_text="FY2010 change", annotation_position="bottom left")
    return base_layout(fig, title, sub, height=640)
PF = f"{base_fy()} dollars per FTE student"


## UC campuses
Total spending by campus (all uses, including medical centers), then core spending per student.

In [2]:
t = add_views(uc.rename(columns={"name": "campus"})[["fy", "campus", "total_k"]].rename(columns={"total_k": "amount_k"}))
order = list(t[t.fy == 2023].sort_values("nominal_b", ascending=False).campus)
stacked_area_toggle(t, "campus", "UC spending by campus", HE + ". Includes medical centers, housing and research.", order=order).show()
keep = [n for n in order if n not in ("UC San Francisco", "UC Law, SF")]
lines(uc, "core_pf", "UC core spending per student, by campus", HE + f". Real dollars per FTE student; excludes hospitals, housing and aid. UC Merced starts FY2006.", keep, PF).show()

## CSU campuses
Core spending per student for each of the 23 campuses (click a name in the legend to hide it, double-click to isolate it).

In [3]:
names = list(csu[csu.fy == 2023].sort_values("core_pf", ascending=False).name)
lines(csu, "core_pf", "CSU core spending per student, by campus", HE + ". Real dollars per FTE student.", names, PF).show()

### Who pays for each CSU campus's students
Latest year: state appropriations and grants per student next to tuition and fees per student, by campus.

In [4]:
z = csu[csu.fy == csu.fy.max()].sort_values("state_pf")
fig = go.Figure()
fig.add_trace(go.Bar(y=z.name, x=z.state_pf, name="State appropriations and grants", orientation="h", marker_color=PALETTE[0], hovertemplate="$%{x:,.0f}"))
fig.add_trace(go.Bar(y=z.name, x=z.tuition_pf, name="Tuition and fees", orientation="h", marker_color=PALETTE[1], hovertemplate="$%{x:,.0f}"))
fig.update_layout(barmode="group", xaxis_title=PF, xaxis_tickprefix="$", hovermode="y unified")
base_layout(fig, f"CSU state support and tuition per student, FY{int(z.fy.max())}", HE + ". Real dollars per FTE student.", height=900)
fig.show()

## Community colleges
The 'unit' is whatever filed (a college or a whole district). Colleges with at least 3,000 full-time-equivalent students in the latest year.

In [5]:
last = cc[cc.fy == cc.fy.max()]
big = last[last.fte >= 3000]
rank = big.sort_values("core_pf")
sel = pd.concat([rank.head(20), rank.tail(20)])
fig = go.Figure(go.Bar(y=sel.name, x=sel.core_pf, orientation="h", marker_color=[PALETTE[2]] * 20 + [PALETTE[0]] * 20, hovertemplate="$%{x:,.0f}"))
fig.update_layout(xaxis_title=PF, xaxis_tickprefix="$", showlegend=False)
base_layout(fig, f"Community colleges: lowest and highest core spending per student, FY{int(last.fy.max())}", HE + f". 20 lowest and 20 highest of {len(big)} colleges with 3,000+ FTE.", height=1000)
fig.show()

### Property taxes versus state money
Community colleges get state money (the Prop 98 apportionment) and a share of local property tax. Colleges in the upper left have large property-tax bases and get less state money per student; the pattern is loose. Dot size is the number of students.

In [6]:
fig = go.Figure(go.Scatter(x=big.state_pf, y=big.local_tax_pf, mode="markers", text=big.name, marker=dict(size=(big.fte / big.fte.max() * 40 + 6), color=PALETTE[3], opacity=0.7, line=dict(width=0.5, color="white")),
                           hovertemplate="%{text}<br>state: $%{x:,.0f}<br>local property tax: $%{y:,.0f}<extra></extra>"))
fig.update_layout(xaxis_title="State appropriations and grants, " + PF, yaxis_title="Local property taxes, " + PF, xaxis_tickprefix="$", yaxis_tickprefix="$", showlegend=False, hovermode="closest")
base_layout(fig, f"Community colleges: state money and local property taxes per student, FY{int(last.fy.max())}", HE + ". Colleges with 3,000+ FTE students.")
fig.show()

### Any college over time
Pick a college or district. Each line is real dollars per full-time-equivalent student.

In [7]:
ids = cc[cc.fy == cc.fy.max()].query("fte >= 3000").sort_values("total_k", ascending=False).name.tolist()
years = list(range(int(cc.fy.min()), int(cc.fy.max()) + 1))
def series(n, col):
    e = cc[cc.name == n].set_index("fy")[col].reindex(years)
    return [None if pd.isna(v) or v in (float("inf"),) else int(round(v)) for v in e]
cols = [("core_pf", "Core spending"), ("state_pf", "State appropriations and grants"), ("local_tax_pf", "Local property taxes"), ("tuition_pf", "Tuition and fees")]
fig = go.Figure()
for i, (c, label) in enumerate(cols):
    fig.add_trace(go.Scatter(x=years, y=series(ids[0], c), name=label, mode="lines+markers", line=dict(color=PALETTE[i]), marker=dict(size=4), hovertemplate="$%{y:,.0f}"))
buttons = [dict(label=n, method="update", args=[{"y": [series(n, c) for c, _ in cols]}, {"title.text": f"{n}: per student<br><sup>{HE}. {PF}.</sup>"}]) for n in ids]
fig.update_layout(updatemenus=[dict(type="dropdown", buttons=buttons, x=0, y=1.0, xanchor="left", yanchor="bottom", active=0, showactive=True)],
                  yaxis_title=PF, yaxis_tickprefix="$", xaxis_title="Fiscal year (ending June 30)")
base_layout(fig, f"{ids[0]}: per student", f"{HE}. {PF}.")
fig.show()

**What the data show** (core spending per full-time-equivalent student, 2026 dollars, FY2005 to FY2023)
- **UC:** the campuses differ by a factor of three or more because medical schools and research sit in this measure. UCLA went from $95k to $155k per student; Berkeley stayed near $65k to $71k; San Diego went from $80k to $96k, Davis $72k to $82k, Irvine $51k to $65k, and the three smallest-budget general campuses (Riverside, Santa Barbara, Santa Cruz) from about $32-34k to $38-40k.
- **CSU:** most campuses are within $15.7k to $22.7k per student in 2023, and many are only a few thousand dollars above 2005. The large urban campuses are lowest (Fullerton $15.7k, Pomona $16.8k, Northridge $17.7k, Los Angeles $17.3k). Small campuses cost more per student: Maritime Academy $64k, Humboldt (now Cal Poly Humboldt) $32k, Channel Islands $30k, Sonoma $29k, Monterey Bay $27k.
- **Community colleges:** among the 90 colleges with 3,000+ students, core spending per student runs from $9.6k (Merritt) to $29k (Mission College) in FY2023. Local property tax per student ranges from almost nothing to $32.5k (Mission) and $23.5k (MiraCosta), with a median of $6.5k, and the two sources tend to move in opposite directions: colleges with little property tax tend to get more per student from the state (Butte $35k and Victor Valley $20k), though the relationship is loose.